# P3 - Snapshot RTI -> datastore AML

Reads the mirrored `telemetry` Delta table from **OneLake**, builds the training
features, and writes a versioned Parquet snapshot into the **ADLS Gen2** account
backing the AML datastore `rti_snapshots`.

```
Eventhouse telemetry --(OneLake availability)--> Delta --> features --> ADLS Gen2 --> AML data asset vN
```

**Auth: no keys anywhere.** The storage account has `allowSharedKeyAccess: false`.
`notebookutils.fs.mount` authenticates with a Microsoft Entra token by default, so
access relies on the *Storage Blob Data Contributor* role already granted to the
user running this notebook. Note that Fabric notebooks do **not** support
`DefaultAzureCredential`.

**ML task** - regression: predict `temperature_c` at `t + 15 min` from the last
30 minutes of all four sensors. Metrics: MAE / RMSE.

In [ ]:
# --- Parameters -------------------------------------------------------------
# These GUIDs are NOT guessed: the OneLake path below is the one published by
# the mirroring policy itself (.show table telemetry policy mirroring ->
# ConnectionStrings). It carries the KQL DATABASE id, not the Eventhouse id.
WORKSPACE_ID = "<FABRIC_WORKSPACE_ID>"
KQL_DATABASE_ID = "<KQL_DATABASE_ID>"
TABLE_NAME = "telemetry"

STORAGE_ACCOUNT = "<STORAGE_ACCOUNT>"
FILESYSTEM = "snapshots"
SNAPSHOT_VERSION = "v1"  # bump to v2 before the retraining demo (P7)

HORIZON_MIN = 15  # prediction horizon
WINDOW_MIN = 30   # rolling feature window
LAGS_MIN = [1, 5, 15]
SENSORS = ["temperature_c", "humidity_pct", "pressure_bar", "vibration_mm_s"]
TARGET = "target_temperature_c"

# Guard rails, expressed in MINUTES PER DEVICE -- the unit that actually matters.
# The raw stream is ~1 msg/s/device, but the 1-minute grid collapses it hard:
# 48k raw rows over 200 min becomes 200 rows per device, not 48k. Sizing the
# guard on raw volume would be meaningless.
# Each device also loses WINDOW_MIN-1 minutes of warm-up and HORIZON_MIN minutes
# of tail, so usable = span - 44 min.
MIN_USABLE_MIN_PER_DEVICE = 120       # hard floor: below this, refuse to train
RECOMMENDED_MIN_PER_DEVICE = 480      # 8 h -> comfortable; get it with
                                      # `python simulator/sensor_producer.py --backfill-minutes 1440`

ONELAKE_PATH = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{KQL_DATABASE_ID}/Tables/{TABLE_NAME}"
)
MOUNT_POINT = "/snapshots"
OUTPUT_PREFIX = f"telemetry/{SNAPSHOT_VERSION}"

print(f"source     : {ONELAKE_PATH}")
print(f"destination: abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{OUTPUT_PREFIX}")


## 1. Read the mirrored table and check it is actually usable

The mirroring policy was created with `"Backfill": false`, meaning only data
ingested **after** OneLake availability was enabled is guaranteed to be mirrored.
Mirroring is also asynchronous (`TargetLatencyInMinutes = 5`). So we verify the
row count and the time span rather than assuming history followed.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = spark.read.format("delta").load(ONELAKE_PATH)

stats = raw.select(
    F.count("*").alias("rows"),
    F.countDistinct("device_id").alias("devices"),
    F.min("timestamp").alias("first_seen"),
    F.max("timestamp").alias("last_seen"),
).collect()[0]

span_min = (stats["last_seen"] - stats["first_seen"]).total_seconds() / 60 if stats["rows"] else 0
print(f"rows       : {stats['rows']:,}")
print(f"devices    : {stats['devices']}")
print(f"first_seen : {stats['first_seen']}")
print(f"last_seen  : {stats['last_seen']}")
print(f"span       : {span_min:.0f} min  (~{span_min / 60:.1f} h)")

if stats["rows"] == 0:
    raise RuntimeError(
        "The Delta mirror is empty. Either mirroring has not caught up yet "
        "(wait one TargetLatencyInMinutes window) or no data has been ingested "
        "since OneLake availability was enabled -- the policy was created with "
        '"Backfill": false, so rows ingested BEFORE that moment may be missing.'
    )

# Compare the mirror against the source of truth. A large negative gap means
# mirroring lag or the Backfill:false effect, not a bug in this notebook.
usable_min = max(span_min - (WINDOW_MIN - 1) - HORIZON_MIN, 0)
print(f"usable     : {usable_min:.0f} min per device after warm-up and tail")

if usable_min < RECOMMENDED_MIN_PER_DEVICE:
    print(
        f"\nNOTE: {usable_min:.0f} usable min/device is below the comfortable "
        f"{RECOMMENDED_MIN_PER_DEVICE} min. Replay more history with:\n"
        f"    python simulator/sensor_producer.py --backfill-minutes 1440\n"
        f"Overlapping minutes are averaged by the 1-minute grid below, so a "
        f"replay does not duplicate rows."
    )


## 2. Feature engineering

Telemetry arrives roughly once per second per device. We first collapse it onto a
**1-minute grid** (mean per sensor): it denoises the signal, makes lags
expressible as row offsets, and matches the 15-minute horizon.

Features per device: lags at 1/5/15 min on all four sensors, rolling mean and
standard deviation over 30 min on temperature and vibration (the load proxy), and
the recent temperature slope. Target: `temperature_c` 15 minutes ahead.

In [ ]:
grid = (
    raw.withColumn("minute", F.date_trunc("minute", F.col("timestamp")))
    .groupBy("device_id", "minute")
    .agg(*[F.avg(c).alias(c) for c in SENSORS])
)

order = Window.partitionBy("device_id").orderBy("minute")

# Row-based lags silently jump over missing minutes, which would fabricate
# relationships that do not exist. Measure the gaps before trusting them.
gaps = (
    grid.withColumn("prev", F.lag("minute").over(order))
    .withColumn(
        "delta_min",
        (F.col("minute").cast("long") - F.col("prev").cast("long")) / 60,
    )
    .filter(F.col("delta_min") > 1)
)
gap_count = gaps.count()
print(f"minutes on the grid : {grid.count():,}")
print(f"gaps > 1 min        : {gap_count}")
if gap_count:
    gaps.select("device_id", "minute", "delta_min").orderBy(F.desc("delta_min")).show(5, False)

In [ ]:
roll = order.rowsBetween(-(WINDOW_MIN - 1), 0)

feat = grid
for sensor in SENSORS:
    for lag in LAGS_MIN:
        feat = feat.withColumn(f"{sensor}_lag{lag}", F.lag(sensor, lag).over(order))

for sensor in ["temperature_c", "vibration_mm_s"]:
    feat = feat.withColumn(f"{sensor}_mean{WINDOW_MIN}", F.avg(sensor).over(roll))
    feat = feat.withColumn(f"{sensor}_std{WINDOW_MIN}", F.stddev(sensor).over(roll))

feat = (
    feat.withColumn(
        "temperature_slope15",
        (F.col("temperature_c") - F.col("temperature_c_lag15")) / 15.0,
    )
    .withColumn("hour_of_day", F.hour("minute"))
    # The label: what the temperature WILL be, HORIZON_MIN rows ahead.
    .withColumn(TARGET, F.lead("temperature_c", HORIZON_MIN).over(order))
)

# Warm-up rows (no lag history) and tail rows (no future label) are incomplete.
dataset = feat.dropna()

feature_columns = [c for c in dataset.columns if c not in ("device_id", "minute", TARGET)]
print(f"feature rows   : {dataset.count():,}")
print(f"feature columns: {len(feature_columns)}")
dataset.select("device_id", "minute", "temperature_c", TARGET).orderBy("minute").show(5, False)

In [ ]:
per_device = dataset.groupBy("device_id").count().orderBy("device_id")
per_device.show()

min_rows = per_device.agg(F.min("count")).collect()[0][0] or 0
if min_rows < MIN_USABLE_MIN_PER_DEVICE:
    raise RuntimeError(
        f"Thinnest device has only {min_rows} usable minutes (floor is "
        f"{MIN_USABLE_MIN_PER_DEVICE}). Replay history first:\n"
        f"    python simulator/sensor_producer.py --backfill-minutes 1440\n"
        f"then wait one mirroring window (5 min) and re-run this notebook. "
        f"Training on this little data yields a model that reports a fine MAE "
        f"and predicts nothing."
    )

# The target must actually differ from the current temperature, otherwise the
# model would just learn the identity function and still look accurate.
dataset.select(
    F.mean(F.abs(F.col(TARGET) - F.col("temperature_c"))).alias("mean_abs_change_15min"),
    F.stddev(TARGET).alias("target_std"),
).show(truncate=False)


## 3. Write the snapshot to the AML datastore

`notebookutils.fs.mount` uses a Microsoft Entra token by default - no account key,
no SAS. If this cell fails with a permission error, the fix is an RBAC grant
(*Storage Blob Data Contributor* on the storage account), never a key.

In [ ]:
import os

import notebookutils

abfss_root = f"abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

if any(m.mountPoint == MOUNT_POINT for m in notebookutils.fs.mounts()):
    notebookutils.fs.unmount(MOUNT_POINT)  # keep the notebook re-runnable
notebookutils.fs.mount(abfss_root, MOUNT_POINT)

local_root = notebookutils.fs.getMountPath(MOUNT_POINT)
output_dir = os.path.join(local_root, "telemetry", SNAPSHOT_VERSION)
os.makedirs(output_dir, exist_ok=True)
print(f"mounted {abfss_root} -> {local_root}")

In [ ]:
# The dataset is small (minute-grain, 4 devices), so a single Parquet file is
# both simpler for AML to consume and avoids the many-small-files problem.
pdf = dataset.orderBy("device_id", "minute").toPandas()

output_file = os.path.join(output_dir, "telemetry_features.parquet")
pdf.to_parquet(output_file, index=False)

print(f"written: {output_file}")
print(f"shape  : {pdf.shape[0]:,} rows x {pdf.shape[1]} columns")
print(f"period : {pdf['minute'].min()} -> {pdf['minute'].max()}")

In [ ]:
notebookutils.fs.unmount(MOUNT_POINT)

print("--- SUMMARY (report in docs/PROGRESS.md) ---")
print(f"snapshot version : {SNAPSHOT_VERSION}")
print(f"rows             : {pdf.shape[0]:,}")
print(f"features         : {len(feature_columns)}")
print(f"target           : {TARGET} (temperature_c at t+{HORIZON_MIN} min)")
print(f"abfss path       : {abfss_root}/{OUTPUT_PREFIX}/telemetry_features.parquet")
print(f"aml datastore uri: azureml://datastores/rti_snapshots/paths/{OUTPUT_PREFIX}")
print()
print("Next: register the AML data asset")
print("  az ml data create -f aml/data/data-asset.yml -g rg-rti-aml-demo -w aml-rti-demo")